# LAB 2 — Profiling GPU con Nsight Systems (NSYS)

**Obiettivo:** usare `nsys` per capire *dove va il tempo* in un programma GPU:
- tempo kernel (GPU)
- tempo memcpy (HtoD / DtoH)
- tempo di attesa lato CPU (`cuCtxSynchronize`)

Per leggere i risultati usiamo un parser ah-hoc “user-friendly” (`tools/parse_nsys.py`) invece dell’output completo di `nsys stats`.


## Struttura repo

```
lab2/
  scripts/
    vecadd.py
    stencil9_profile.py
    stencil49_profile.py
    sync_antipattern.py
  tools/
    parse_nsys.py
  results/
    nsys/
  LAB2.ipynb
```


## 0) Setup e requisiti

Serve:
- Linux + GPU NVIDIA + driver
- Python env con `numba`, `numpy`
- Nsight Systems (`nsys`) disponibile sul server

L'ambiente dovrebbe essere gia' installato dopo il lab 1. 
Se serve, vedi le istruzioni di setup nel notebook `lab1`.


In [1]:
# controlliamo che la GPU sia disponibile
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Tue_Feb_27_16:19:38_PST_2024
Cuda compilation tools, release 12.4, V12.4.99
Build cuda_12.4.r12.4/compiler.33961263_0


In [2]:
# Wrap your Numba code in a function or a script
def run_my_numba_code():
    from numba import cuda
    import numpy as np
    
    @cuda.jit
    def add_kernel(a, b, c):
        i = cuda.grid(1)
        if i < a.size:
            c[i] = a[i] + b[i]

    N = 10**6
    a = cuda.to_device(np.ones(N, dtype=np.float32))
    b = cuda.to_device(np.ones(N, dtype=np.float32))
    c = cuda.device_array_like(a)

    add_kernel[3907, 256](a, b, c)
    cuda.synchronize()

# Save the code to a temporary file
with open('profile_me.py', 'w') as f:
    f.write("""
import numpy as np
from numba import cuda
@cuda.jit
def add_kernel(a, b, c):
    i = cuda.grid(1)
    if i < a.size:
        c[i] = a[i] + b[i]

N = 10**6
a = cuda.to_device(np.ones(N, dtype=np.float32))
b = cuda.to_device(np.ones(N, dtype=np.float32))
c = cuda.device_array_like(a)
add_kernel[3907, 256](a, b, c)
cuda.synchronize()
    """)

# Run the profile command
!nsys profile -o my_numba_report --force-overwrite=true python profile_me.py

Generating '/tmp/nsys-report-a356.qdstrm'
[1/1] [========================100%] my_numba_report.nsys-rep
Generated:
    /davinci-1/home/glisita/Projects/CudaOpt/Module2-Optimization/lab2/lab2/my_numba_report.nsys-rep


In [3]:
# Verifiche base
!nvidia-smi
!python -c "import numpy as np; import numba; from numba import cuda; print('Numba:', numba.__version__); print('GPU:', cuda.get_current_device().name)"
!nsys --version

Mon Feb 23 12:28:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:2F:00.0 Off |                    0 |
| N/A   24C    P0             50W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1) Comandi da usare

### Profilare da linea di comando (genera `.nsys-rep`)
```bash
!mkdir -p results/nsys
!rm -f results/nsys/exp.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/exp python scripts/EXP.py
```

### Riassunto “pulito”, basato sul parser custom in Python (`tools/parse_nsys.py`)
```bash
python tools/parse_nsys.py results/nsys/exp.nsys-rep
```
l’output di `nsys stats` è lungo: lo useremo solo come approfondimento.


## 2) TODO 1 — Scrivere un kernel semplicissimo: Vector Add

Script: `scripts/vecadd.py`

**Cosa fare:**
- Completare il kernel GPU per sommare 2 vettori (`C = A + B`)
- Eseguire e profilare


In [4]:
!mkdir -p results/nsys
!rm -f results/nsys/exp.nsys-rep

In [5]:
# Run (no profiling)
!python ./scripts/vec_add.py

vec_add correct=True | n=10000000 | grid=39063 block=256


In [6]:
# Profiling + summary
!rm -f ./results/nsys/vecadd.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o ./results/nsys/vecadd python ./scripts/vec_add.py
!python ./tools/parse_nsys.py ./results/nsys/vecadd.nsys-rep

vec_add correct=True | n=10000000 | grid=39063 block=256
Generating '/tmp/nsys-report-d915.qdstrm'
[1/1] [========================100%] vecadd.nsys-rep
Generated:
    /davinci-1/home/glisita/Projects/CudaOpt/Module2-Optimization/lab2/lab2/./results/nsys/vecadd.nsys-rep
=== NSYS SUMMARY (student-friendly) ===
CPU wait (cuCtxSynchronize): 0.140 ms
GPU kernel total           : 0.187 ms
GPU memcpy total           : 12.397 ms (HtoD 5.809 ms, DtoH 6.588 ms)

Top 1 kernels by total GPU time:
  1. vec_add
     total: 0.187 ms | instances: 2
     avg per launch: 0.094 ms


`nsys stats`: output completo (non useremo ora)


In [11]:
!nsys stats --force-export=true ./results/nsys/vecadd.nsys-rep

Generating SQLite file ./results/nsys/vecadd.sqlite from ./results/nsys/vecadd.nsys-rep
Processing [./results/nsys/vecadd.sqlite] with [/opt/nvidia/nsight-systems/2025.3.2/host-linux-armv8/reports/nvtx_sum.py]... 
SKIPPED: ./results/nsys/vecadd.sqlite does not contain NV Tools Extension (NVTX) data.

Processing [./results/nsys/vecadd.sqlite] with [/opt/nvidia/nsight-systems/2025.3.2/host-linux-armv8/reports/osrt_sum.py]... 

 ** OS Runtime Summary (osrt_sum):

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)    Med (ns)   Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  ---------  --------  -----------  ------------  ----------------------
     60.7      806,152,704        183   4,405,206.0  884,272.0     1,984  150,246,992  13,584,885.3  poll                  
     24.6      326,313,568         19  17,174,398.3  489,136.0   100,160  311,328,480  71,240,604.1  sem_wait              
     13.5      179,751,760        438  

## 3) TO-DO 2 — Matrix Multiplication CUDA: naive vs tiled (shared memory)

L'obiettivo e' implementere due versioni della moltiplicazione di matrici quadrate `C = A × B` e profilare le implementazioni GPU con **Nsight Systems**. Le versioni da implementare sono:

1. una versione **CPU** di riferimento ( `matmul_cpu(A, B)` )
2. una versione **CUDA naive** (solo global memory)
3. una versione **CUDA tiled** con **shared memory**

E' richiesto di completare il file `matmul_profile.py` con le implementazioni mancanti, verificare la correttezza dei risultati e profilare con `nsys` per confrontare i tempi di esecuzione.

### Parte 1 — Implementazione CPU (TODO)

Scrivere una funzione CPU che calcoli:

```math
C[i,j] = \sum_k A[i,k] \cdot B[k,j]
```

Requisiti:
- input: `A, B` NumPy `float32` di dimensione `n×n`
- output: `C` NumPy `float32`
- usare **due o tre cicli annidati**
- NON usare `A @ B` o funzioni BLAS

---

### Parte 2 — Kernel CUDA naive (TODO)

Scrivere un kernel `@cuda.jit matmul_naive(...)` CUDA che:
- usa una **griglia 2D** (`cuda.grid(2)`)
- ogni thread calcola **un elemento** `C[i,j]`
- legge `A` e `B` **solo dalla memoria globale**
- non usa shared memory

---

### Parte 3 — Kernel CUDA tiled con shared memory (TODO)

Scrivere un kernel CUDA che:
- divide le matrici in **tile quadrati** con tile size `TILE × TILE` ( `block = (TILE, TILE)`)
- carica porzioni di `A` e `B` in **shared memory**
- sincronizza i thread del blocco (`cuda.syncthreads()`)
- accumula il risultato parziale

---

### Parte 4 — Verifica di correttezza

Per ogni implementazione:
- confrontare il risultato GPU con la versione CPU
- usare `np.allclose` con tolleranze adeguate per `float32`
- riportare il massimo errore assoluto osservato

---

### Parte 5 — Benchmark e profiling

Profilare il programma con **Nsight Systems** e misurare:
- tempo CPU (una sola esecuzione)
- tempo GPU naive (media su più iterazioni)
- tempo GPU tiled (media su più iterazioni)


```bash
nsys profile -t cuda,nvtx,osrt -o results/nsys/matmul python scripts/matmul_profile.py


In [ ]:
!rm -f ./results/nsys/matmul_profile.py
!nsys profile --force-overwrite true -t cuda,osrt -o ./results/nsys/matmul_profile python ./scripts/matmul_profile.py
!python ./tools/parse_nsys.py ./results/nsys/matmul_profile.nsys-rep

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

Numba CUDA MatMul profiling script (+ optional CuPy/cuBLAS)
GPU: NVIDIA GB10
TILE = 16
CuPy available: True

=== n = 512 ===
CPU (NumPy BLAS)  : 4.61 ms (single run)
GPU naive         : 1.46 ms/iter | correct=True | max_abs_err=0.0001068
GPU tiled(shared) : 1.42 ms/iter | correct=True | max_abs_err=0.0001068
Speedup tiled vs naive: 1.03x
GPU CuPy (cuBLAS) : 0.05 ms/iter | correct=True | max_abs_err=0.0001144
Speedup cuBLAS vs tiled: 30.38x

=== n = 1024 ===
CPU (NumPy BLAS)  : 3.27 ms (single run)
GPU naive         : 11.28 ms/iter | correct=True | max_abs_err=0.0002136
GPU tiled(shared) : 11.23 ms/iter | correct=True | max_abs_err=0.0002136
Speedup tiled vs naive: 1.00x
GPU CuPy (cuBLAS) : 0.14 ms/iter | correct=True | max_abs_err=0.0001068
Speedup cuBLAS vs tiled: 82.92x

=== n = 2048 ===
CPU (NumPy BLAS)  : 28.46 ms (single run)
GPU naive         : 88.96 ms/iter | c

In [25]:
!nsys stats --force-export=true ./results/nsys/matmul_profile.nsys-rep

Generating SQLite file ./results/nsys/matmul_profile.sqlite from ./results/nsys/matmul_profile.nsys-rep
Processing [./results/nsys/matmul_profile.sqlite] with [/opt/nvidia/nsight-systems/2025.3.2/host-linux-armv8/reports/nvtx_sum.py]... 
SKIPPED: ./results/nsys/matmul_profile.sqlite does not contain NV Tools Extension (NVTX) data.

Processing [./results/nsys/matmul_profile.sqlite] with [/opt/nvidia/nsight-systems/2025.3.2/host-linux-armv8/reports/osrt_sum.py]... 

 ** OS Runtime Summary (osrt_sum):

 Time (%)  Total Time (ns)  Num Calls     Avg (ns)         Med (ns)        Min (ns)        Max (ns)       StdDev (ns)             Name         
 --------  ---------------  ---------  ---------------  ---------------  -------------  --------------  ---------------  ----------------------
     47.7  157,283,390,912         77  2,042,641,440.4    490,093,728.0        678,496   6,524,198,656  2,579,651,832.3  pthread_cond_wait     
     17.8   58,808,319,056     10,900      5,395,258.6      2,9

## 4) TO-DO 3: — Riduzione: `sum/mean` (CuPy vs Numba)

In questo esercizio confrontiamo una riduzione ottimizzata (CuPy) con una riduzione “naive” scritta con Numba CUDA. Le riduzioni sono difficili da ottimizzare e perché una libreria (CuPy) può essere molto più veloce di un kernel scritto “a mano” senza ottimizzazioni avanzate. Il codice si trova in `scripts/reduction_profile.py`.

---

### Task A — CuPy

Dato un vettore 1D `x` (float32, molto grande, es. `N = 50_000_000`):

1. Calcolare `s_cupy = cp.sum(x) \ cp.mean(x)`.
2. Misurare il tempo medio su almeno 30 iterazioni (con warmup).
3. Sincronizzare correttamente la GPU prima di fermare il timer.

---

### Task B — Numba naive (partial sums per blocco)

Implementare una riduzione in due fasi:

1. **Kernel CUDA** `block_sum_kernel(x, partial)`:

   * ogni blocco accumula una somma parziale in `partial[blockIdx.x]`
   * usare `cuda.grid(1)` e un loop a stride per coprire tutto `x`

2. **Finalizzazione su CPU**:

   * copiare `partial` su host
   * sommare su CPU per ottenere `s_numba`

---

### Verifiche

* Confrontare entrambe le somme con un riferimento CPU accurato: `np.sum(x_host, dtype=np.float64) \ np.mean(x_host, dtype=np.float64)`.
* Calcolare errore relativo:
   ```math
  \frac{|s - s_{ref}|}{|s_{ref}|}
   ```




In [17]:
!rm -f ./results/nsys/reduction_profile.py
!nsys profile --force-overwrite true -t cuda,osrt -o ./results/nsys/reduction_profile python ./scripts/reduction_profile.py
!python ./tools/parse_nsys.py ./results/nsys/reduction_profile.nsys-rep

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

/home/e4user/Documents/s362027_6-02-26/cuda_lab/lib/python3.12/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/e4user/Documents/s362027_6-02-26/cuda_lab/lib/python3.12/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/e4user/Documents/s362027_6-02-26/cuda_lab/lib/python3.12/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
=== Reduction: r = sum(x) / mean(x) ===
N = 50000000
CPU ref (float64)       : r=50000000.000000

CuPy (sum+mean)         : 1.984 ms/iter |

## 5) Stencil kernel

Uno stencil è un’operazione che calcola ogni elemento di una matrice usando i **valori vicini**.

Gli stencil sono molto comuni in:

* image processing
* computer vision
* simulazioni fisiche
* AI (pre-processing, convoluzioni)

---

## Stencil 3×3 (9-point)

Per ogni elemento `(y, x)` della matrice di input:

* legge i **9 valori** in un intorno 3×3
* calcola la **media**
* scrive il risultato nella matrice di output

Visivamente:

```
x x x
x O x
x x x
```

Dove `O` è il punto calcolato e `x` sono i vicini.

### Perché lo usiamo

* è **semplice**
* ha **poco riuso dei dati**
* spesso la cache GPU è già sufficiente

Se il riuso è basso, l’overhead può peggiorare le prestazioni.

Nel laboratorio:

* la versione “shared” può essere **uguale o più lenta** della naive

---

## Stencil 7×7 (49-point)

### Cosa fa

Per ogni elemento `(y, x)`:

* legge i **49 valori** in un intorno 7×7
* calcola la **media**
* scrive il risultato

Visivamente:

```
x x x x x x x
x x x x x x x
x x x x x x x
x x x O x x x
x x x x x x x
x x x x x x x
x x x x x x x
```

### Perché è diverso dal 3×3

* ogni output legge **molti più dati**
* il costo delle letture di memoria è molto più alto
* il **riuso dei dati** tra thread aumenta

> Quando il riuso dei dati è alto, caricare un blocco in shared memory può diventare conveniente.

Nel laboratorio:

* la versione “shared” può mostrare un **miglioramento misurabile** (ma non sempre...)
## Versione naive vs versione shared

### Versione naive

* ogni thread legge tutti i dati direttamente dalla **global memory**
* codice semplice
* zero sincronizzazioni
* nessuna cooperazione tra thread

### Versione shared

* i thread di un blocco **collaborano**
* caricano una porzione della matrice in **shared memory**
* riusano i dati più volte
* richiede:

  * logica aggiuntiva
  * sincronizzazione (`cuda.syncthreads()`)

La shared memory **non è sempre migliore**, ma conviene quando il riuso dei dati compensa l’overhead.


### Parte 1 — Shared memory NON sempre aiuta (3×3 / 9-point)

Script: `scripts/stencil9_profile.py`

**Obiettivo:** vedere un caso in cui la versione “shared” può essere uguale o peggiore della naive (overhead > beneficio).

Esegui lo script e annota `GPU naive` e `GPU shared`:



In [18]:
!python scripts/stencil9_profile.py

Numba CUDA 9-point stencil (3x3 blur) profiling script
GPU: NVIDIA GB10
Block naive  : (32, 8)
Block shared : (32, 8)  shared tile = (10, 34) with halo=1

=== size = 2048 x 2048 ===
CPU ref (NumPy)      : 11.38 ms (single run)
GPU naive            : 0.4278 ms/iter | correct=True | max_abs_err=1.79e-07
GPU shared(tiling)   : 0.5038 ms/iter | correct=True | max_abs_err=1.79e-07
Speedup shared vs naive: 0.85x

=== size = 4096 x 4096 ===
CPU ref (NumPy)      : 47.55 ms (single run)
GPU naive            : 1.7710 ms/iter | correct=True | max_abs_err=1.79e-07
GPU shared(tiling)   : 1.9340 ms/iter | correct=True | max_abs_err=1.79e-07
Speedup shared vs naive: 0.92x



In [19]:
!rm -f results/nsys/stencil9.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/stencil9 python scripts/stencil9_profile.py
!python tools/parse_nsys.py results/nsys/stencil9.nsys-rep

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

Numba CUDA 9-point stencil (3x3 blur) profiling script
GPU: NVIDIA GB10
Block naive  : (32, 8)
Block shared : (32, 8)  shared tile = (10, 34) with halo=1

=== size = 2048 x 2048 ===
CPU ref (NumPy)      : 14.97 ms (single run)
GPU naive            : 0.4452 ms/iter | correct=True | max_abs_err=1.79e-07
GPU shared(tiling)   : 0.4979 ms/iter | correct=True | max_abs_err=1.79e-07
Speedup shared vs naive: 0.89x

=== size = 4096 x 4096 ===
CPU ref (NumPy)      : 62.36 ms (single run)
GPU naive            : 1.7702 ms/iter | correct=True | max_abs_err=1.79e-07
GPU shared(tiling)   : 1.9184 ms/iter | correct=True | max_abs_err=1.79e-07
Speedup shared vs naive: 0.92x

Generating '/tmp/nsys-report-31b1.qdstrm'
[1/1] [========================100%] stencil9.nsys-rep
Generated:
	/home/e4user/Documents/s362027_6-02-26/lab2/lab2/results/nsys/stencil9.nsys-rep
=== NSYS SUMMARY (studen

**Domande:**
- Quale kernel è più veloce?
- Le memcpy contano o no?
- `cuCtxSynchronize` domina? perché?

### Parte 2 — Shared memory può aiutare (7×7 / 49-point)

Script: `scripts/stencil49_profile.py`

**Obiettivo:** vedere un caso in cui la versione “shared” ha senso (riuso alto).

In [20]:
# Run (no profiling)
!python scripts/stencil49_profile.py --mode both

Numba CUDA 49-point stencil (7x7 blur)
GPU: NVIDIA GB10
Block naive  : (32, 8)
Block shared : (32, 8)  shared tile = (14, 38)

=== size = 2048 x 2048 ===
GPU naive  : 2.138 ms | correct=True
GPU shared : 2.204 ms | correct=True
Speedup shared vs naive: 0.97x

=== size = 4096 x 4096 ===
GPU naive  : 8.644 ms | correct=True
GPU shared : 8.657 ms | correct=True
Speedup shared vs naive: 1.00x



In [21]:
# Profile naive
!rm -f results/nsys/stencil49_naive.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/stencil49_naive python scripts/stencil49_profile.py
!python tools/parse_nsys.py results/nsys/stencil49_naive.nsys-rep

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

Numba CUDA 49-point stencil (7x7 blur)
GPU: NVIDIA GB10
Block naive  : (32, 8)
Block shared : (32, 8)  shared tile = (14, 38)

=== size = 2048 x 2048 ===
GPU naive  : 2.205 ms | correct=True
GPU shared : 2.201 ms | correct=True
Speedup shared vs naive: 1.00x

=== size = 4096 x 4096 ===
GPU naive  : 8.847 ms | correct=True
GPU shared : 8.825 ms | correct=True
Speedup shared vs naive: 1.00x

Generating '/tmp/nsys-report-3c2f.qdstrm'
[1/1] [========================100%] stencil49_naive.nsys-rep
Generated:
	/home/e4user/Documents/s362027_6-02-26/lab2/lab2/results/nsys/stencil49_naive.nsys-rep
=== NSYS SUMMARY (student-friendly) ===
CPU wait (cuCtxSynchronize): 2421.454 ms
GPU kernel total           : 2422.747 ms
GPU memcpy total           : 119.275 ms (HtoD 1.406 ms, DtoH 117.870 ms)

Top 2 kernels by total GPU time:
  1. stencil49_naive
     total: 1213.133 ms | instance

**Domande:**
- Quale kernel è più veloce?
- Le memcpy contano o no?
- `cuCtxSynchronize` domina? perché?


## 5) TO-DO 3 - Anti-pattern: sincronizzare nel posto sbagliato

Script: `scripts/sync_antipattern.py`

**Obiettivo:** far vedere che `cuda.synchronize()` dentro al loop serializza tutto e aumenta l’attesa CPU.

Useremo:
- `--mode good`
- `--mode bad`

Completare, profilare e confrontare i summary delle due versioni


### Anti-pattern: sincronizzare nel posto sbagliato (`cuda.synchronize()`)

### Cos’è una sincronizzazione

`cuda.synchronize()` forza la **CPU ad aspettare** finché la GPU non ha terminato **tutto** il lavoro lanciato fino a quel punto.

Normalmente:

* la CPU **lancia** un kernel GPU
* la GPU **lavora in parallelo**
* la CPU può continuare a fare altro

Nella versione “good”:

* i kernel vengono lanciati in un loop
* **una sola sincronizzazione** viene fatta alla fine, per misurare il tempo totale

Pseudo-codice:

```python
for i in range(N):
    launch_kernel()
cuda.synchronize()   # una sola volta
```

la GPU lavora in modo continuo, mentre la CPU puo' continuare a fare altro. La pipeline è efficiente

## Anti-pattern (BAD): sincronizzare dentro al loop

Nella versione “bad”:

* la CPU lancia il kernel
* **subito dopo si ferma ad aspettare**
* questo succede ad ogni iterazione

Pseudo-codice:

```python
for i in range(N):
    launch_kernel()
    cuda.synchronize()   # ANTI-PATTERN
```


la GPU viene usata in modo **seriale**. Si perde completamente il vantaggio dell’asincronia.

---

Nel profiling con Nsight Systems:

* la riga `cuCtxSynchronize` **domina il tempo totale**
* il tempo kernel può essere identico tra GOOD e BAD
* ma il tempo totale peggiora

In applicazioni reali (AI, vision, simulazioni),  sincronizzare troppo spesso è un anti-pattern comune che: riduce throughput, aumenta latenza, spreca GPU costose

In [22]:
# Run (no profiling)
!python scripts/sync_antipattern.py --mode good
!python scripts/sync_antipattern.py --mode bad

mode=good | 0.017 ms/iter | correct=True | n=2000000
mode=bad | 0.051 ms/iter | correct=True | n=2000000


In [23]:
# Profile GOOD
!rm -f results/nsys/sync_good.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/sync_good python scripts/sync_antipattern.py --mode good
!python tools/parse_nsys.py results/nsys/sync_good.nsys-rep

# Profile BAD
!rm -f results/nsys/sync_bad.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/sync_bad python scripts/sync_antipattern.py --mode bad
!python tools/parse_nsys.py results/nsys/sync_bad.nsys-rep

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

mode=good | 0.017 ms/iter | correct=True | n=2000000
Generating '/tmp/nsys-report-6f6b.qdstrm'
[1/1] [========================100%] sync_good.nsys-rep
Generated:
	/home/e4user/Documents/s362027_6-02-26/lab2/lab2/results/nsys/sync_good.nsys-rep
=== NSYS SUMMARY (student-friendly) ===
CPU wait (cuCtxSynchronize): 0.638 ms
GPU kernel total           : 127.487 ms
GPU memcpy total           : 0.412 ms (HtoD 0.278 ms, DtoH 0.134 ms)

Top 1 kernels by total GPU time:
  1. saxpy
     total: 127.487 ms | instances: 20020
     avg per launch: 0.006 ms
Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

mode=bad | 0.064 ms/iter | correct=True | n=2000000
Generating '/tmp/nsys-report-f2c0.qdstrm'
[1/1] [========================100%] sync_bad.nsys-rep
Generated:
	/home/e4user/Documents/s362027_6-02-26/lab2/lab2/re

## 6) Block size sweep (stencil49)

`scripts/stencil49_sweep.py`

Questo script serve a **testare diverse configurazioni di block size** per lo stesso kernel GPU (`stencil49_shared`, blur 7×7).

Il kernel **non cambia**: cambia solo **come viene lanciato** sulla GPU
(numero di thread per blocco in X e Y).

Questo esercizio mostra che **la launch configuration è parte dell’ottimizzazione**.

---

### Cosa fa lo script

`stencil49_sweep.py` fa due cose distinte:

1. **Sweep con timing (veloce)**
   Prova più configurazioni di block size e stampa una **tabella comparativa**:

   * tempo medio per iterazione (ms)
   * speedup rispetto al kernel naive

2. **Profiling pulito con NSYS (opzionale)**
   Permette di profilare **una sola configurazione scelta**, evitando report confusi.

---

### Step A — Sweep veloce (solo timing)

Esegui lo sweep e guarda la tabella finale
(**ms/iter più basso = meglio**):

```bash
python scripts/stencil49_sweep.py --h 2048 --w 2048
```

Per default lo script prova queste configurazioni (BX×BY):

* `32×8`  (baseline)
* `32×16`
* `16×16`

---

### TODO

Aggiungi **1–2 configurazioni tue** (es. `64×4`, `8×32`) passando l’argomento `--configs`:

```bash
python scripts/stencil49_sweep.py --h 2048 --w 2048 \
  --configs "32x8,32x16,16x16,64x4,8x32"
```

Annota:

* quale configurazione è la più veloce
* di quanto migliora rispetto alla baseline

---

### Step B — Profiling con NSYS (solo la configurazione migliore)

Scegli **una sola** configurazione dallo sweep (esempio: `16×16`)
e profila **solo quella**, per avere un report chiaro:

```bash
rm -f results/nsys/st49_best.nsys-rep
nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/st49_best \
  python scripts/stencil49_sweep.py --h 2048 --w 2048 --block 16,16 --only-shared
python tools/parse_nsys.py results/nsys/st49_best.nsys-rep
```

---

### Cosa osservare nel riassunto NSYS

* **tempo totale kernel GPU**
* **tempo di attesa CPU (`cuCtxSynchronize`)**
* assenza di memcpy dominanti (in questo caso)

---

### Cosa consegnare

1. tabella dello sweep (copiata dall’output del terminale)
2. block size scelto come “best”
3. output del parser NSYS per la configurazione best

---


# Tabella dello sweep

### System Configuration
*   **GPU:** NVIDIA GB10
*   **Stencil:** 7x7 (radius=3)
*   **Domain Size:** 2048 $\times$ 2048
*   **Execution:** 120 iterations (20 warmup)

---

### Baseline: Naive Implementation
*   **Block Dimensions:** (32, 8)
*   **Time:** 1.941 ms/iter
*   **Status:** Correct

---

### Shared Kernel Block Sweep
*Lower time is better.*

| BX | BY | Time (ms/iter) | Speedup (vs Naive) | Correct |
| :---: | :---: | :---: | :---: | :---: |
| **16** | **16** | **1.934** | **1.00** | **True** |
| 64 | 4 | 1.936 | 1.00 | True |
| 32 | 8 | 1.938 | 1.00 | True |
| 32 | 16 | 1.946 | 1.00 | True |

# Block size scelto come “best”

python3 scripts/stencil49_sweep.py --h 2048 --w 2048 --configs '16x16'

### System Configuration
*   **GPU:** NVIDIA GB10
*   **Stencil:** 7x7 (radius=3)
*   **Domain Size:** 2048 $\times$ 2048
*   **Execution:** 120 iterations (20 warmup)

---

### Baseline: Naive Implementation
*   **Block Dimensions:** (32, 8)
*   **Time:** 1.940 ms/iter
*   **Status:** Correct

---

### Shared Kernel Block Sweep
*Lower time is better.*

| BX | BY | Time (ms/iter) | Speedup (vs Naive) | Correct |
| :---: | :---: | :---: | :---: | :---: |
| 16 | 16 | 1.945 | 1.00 | True |


# Output del parser NSYS per la configurazione best

=== NSYS SUMMARY (student-friendly) ===
CPU wait (cuCtxSynchronize): 270.211 ms
GPU kernel total           : 270.242 ms
GPU memcpy total           : 0.289 ms (HtoD 0.289 ms, DtoH 0.000 ms)

Top 1 kernels by total GPU time:
  1. stencil49_shared
     total: 270.242 ms | instances: 140
     avg per launch: 1.930 ms

# 8. Conclusioni, Domande aperte e Consegna

Il notebook di lavoro del lab deve essere consegnato nella sezione 'elaborati' sul portale della didattica. Questo deve contenere il codice compeleto e i grafici richiesti. Sono richiesti gli output di tempi e correctness per:
- vector add (TODO 1)
- matmul naive vs tiled (TODO 2)
- stencil9
- stencil49 (naive vs shared)
- sync antipattern (good vs bad)

Costruire una tabella con:

| Componente        | Tempo (ms) | Percentuale |
| ----------------- | ---------- | ----------- |
| Kernel CUDA       |            |             |
| Trasferimenti H→D |            |             |
| Trasferimenti D→H |            |             |
| Altro / CPU       |            |             |

Scrivere **5–10 righe** che descrivano:

* quale parte domina il tempo di esecuzione
* se il programma è:
  * compute-bound
  * memory-bound
  * transfer-bound
---

Rispondere alle seguenti domande (brevemente, indicativamente 3-5 righe per domanda):
- Cosa indica `cuCtxSynchronize`?
Indica una sincronizzazione esplicita del contesto GPU, la CPU attende che tutte le operazioni GPU completino prima di continuare.
È utile per misurare il tempo effettivo di esecuzione.
- Shared Memory: quando puo' aiutare?
Quando i dati vengono riusati frequentemente da diversi thread dello stesso block ed i pattern di accesso alla memoria globale sono regolari  (data coalescence).
Nel caso dello stencil 7x7: la shared memory potrebbe ridurre gli accessi a global memory per elementi contigui
- Qual è il bottleneck principale osservato nel report NSYS?
- Il tempo speso nei trasferimenti è significativo rispetto al calcolo? Perché?
- I kernel CUDA sono lanciati in modo continuo o frammentato nel tempo?
- In base ai report, quale sarebbe la **prima ottimizzazione** da tentare?

E' inoltre richiesto di completare i seguenti esercizi


## Esercizio 1 — Operazione elementwise 1D (Numba vs CuPy)

Implementare l’operazione elementwise:

```math
y[i] = \sin(x[i]) + x[i]^2
```

dove `x` e `y` sono array 1D `float32`.

---

### Parte A — Implementazione Numba

1. Scrivere un kernel CUDA con Numba (`@cuda.jit`) in un file denominato `elem1d_todo_profile.py`.
2. Il kernel deve:
   * supportare array di lunghezza arbitraria;
   * usare `cuda.grid(1)` e un **loop a stride** (come visto nel Lab 1).
3. Scrivere una funzione host che:
   * copi i dati su GPU,
   * lanci il kernel,
   * riporti il risultato su host.
4. Verificare la correttezza confrontando il risultato GPU con:

   ```python
   np.sin(x) + x**2
   ```

   usando `np.allclose`.

---

### Parte B — Implementazione CuPy

5. Implementare la stessa operazione utilizzando **solo CuPy**, senza scrivere kernel CUDA espliciti.
6. L’implementazione deve eseguire l’operazione con funzioni CuPy  (`y_d[...] = cp.sin(x_d) + x_d * x_d` ATTENZIONE: qui `y_d[...]` è corretto e parte dell’implementazione, non è un placeholder)
7. Scrivere una funzione host che:
   * trasferisca i dati su GPU,
   * esegua l’operazione con funzioni CuPy,
   * riporti il risultato su host.
7. Verificare la correttezza confrontando con la versione NumPy.

---

### Parte C — Profilazione e confronto

8. Profilare entrambe le versioni (Numba e CuPy) con **NSYS**.
9. Costruire una tabella che riporti:

| Metodo | Tempo kernel | 
| ------ | ------------ |
| Numba  |              |
| CuPy   |              |
---

## Esercizio 2 — Stencil 2D a 5 punti (Numba vs CuPy)

Dato un input `A` di dimensione **H×W**, calcolare un output `B` di dimensione **(H−2)×(W−2)**:

```math
B[i,j] = A[i,j] + A[i-1,j] + A[i+1,j] + A[i,j-1] + A[i,j+1]
```

---

### Parte A — Implementazione Numba

1. Scrivere un kernel CUDA 2D con Numba (`@cuda.jit`) usando `cuda.grid(2)`.
   Ogni thread deve calcolare un singolo elemento di `B`.
2. Scrivere una funzione host che:
   * allochi input e output su GPU,
   * scelga una configurazione 2D di thread per block,
   * lanci il kernel,
   * riporti il risultato su host.
3. Verificare la correttezza confrontando con una versione NumPy su una matrice **512×512** (`np.allclose`).

---

### Parte B — Implementazione CuPy

4. Implementare lo stesso stencil utilizzando **solo CuPy**, sfruttando slicing e operazioni elementwise.
5. L’output deve avere dimensione **(H−2)×(W−2)**.
6. Verificare la correttezza confrontando con la versione NumPy.

---

### Parte C — Profilazione e confronto

7. Profilare entrambe le versioni (Numba e CuPy) su una matrice grande (es. **4096×4096**).
8. Costruire una tabella che riporti:

| Metodo | Tempo kernel | 
| ------ | ------------ |
| Numba  |              |
| CuPy   |              |


In [16]:
# Profile TODO elem1d
!rm -f results/nsys/elem1d_todo_profile.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/elem1d_todo_profile python scripts/elem1d_todo_profile.py
!python tools/parse_nsys.py results/nsys/elem1d_todo_profile.nsys-rep


Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

Elem1D max abs diff (numba vs cpu): 1.1920929e-07
Elem1D max abs diff (cupy  vs cpu): 1.1920929e-07
CPU : 0.0180s
Numba (incl one D2H): 0.2624s
CuPy  (incl one D2H): 0.0046s
Generating '/tmp/nsys-report-5189.qdstrm'
[1/1] [========================100%] elem1d_todo_profile.nsys-rep
Generated:
	/home/e4user/Downloads/master_ai/SOLUZIONE/lab2/results/nsys/elem1d_todo_profile.nsys-rep
=== NSYS SUMMARY (student-friendly) ===
CPU wait (cuCtxSynchronize): 2.439 ms
GPU kernel total           : 8.646 ms
GPU memcpy total           : 263.296 ms (HtoD 1.374 ms, DtoH 261.923 ms)

Top 3 kernels by total GPU time:
  1. cupy_add__float32_float32_float32
     total: 2.710 ms | instances: 6
     avg per launch: 0.452 ms
  2. cupy_multiply__float32_float32_float32
     total: 1.991 ms | instances: 6
     avg per launch: 0.332 ms
  3. elem1d_kernel
     total: 1.985 ms | instances: 6
   

In [17]:
# Profile TODO stencil5
!rm -f results/nsys/stencil5_todo_profile.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/stencil5_todo_profile python scripts/stencil5_todo_profile.py
!python tools/parse_nsys.py results/nsys/stencil5_todo_profile.nsys-rep

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

Stencil5 max abs diff (numba vs cpu): 0.0
Stencil5 max abs diff (cupy  vs cpu): 0.0
Stencil5 Numba (4096x4096, incl copies): 0.4424s | out=(4094, 4094)
Stencil5 CuPy  (4096x4096, incl copies): 0.0387s | out=(4094, 4094)
Generating '/tmp/nsys-report-17a2.qdstrm'
[1/1] [========================100%] stencil5_todo_profile.nsys-rep
Generated:
	/home/e4user/Downloads/master_ai/SOLUZIONE/lab2/results/nsys/stencil5_todo_profile.nsys-rep
=== NSYS SUMMARY (student-friendly) ===
CPU wait (cuCtxSynchronize): 2.145 ms
GPU kernel total           : 4.182 ms
GPU memcpy total           : 441.271 ms (HtoD 2.453 ms, DtoH 438.817 ms)

Top 2 kernels by total GPU time:
  1. cupy_add__float32_float32_float32
     total: 3.120 ms | instances: 8
     avg per launch: 0.390 ms
  2. stencil5_kernel
     total: 1.062 ms | instances: 2
     avg per launch: 0.531 ms
